# 🎯 [AMM02_M04] 적 저항 수준 및 화력 운용 계획 연계 일자별 소모량 시계열 예측 모델 고도화 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/AMM02/amm02_m04_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 탄약·특수물자 (Level 3) |
| **미션 ID** | AMM02_M04 |
| **미션 주제** | 적 저항 수준 및 화력 운용 계획 연계 일자별 소모량 시계열 예측 모델 고도화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
탄약 저장고는 공간이 한정되어 있고 품목마다 함께 둘 수 없는 제약이 있어, 앞으로 얼마나 소모되고 얼마나 자주 반출될지를 알아야 입고 위치와 재보급 순서를 합리적으로 정할 수 있다. 그러나 작전 단계별 고정 기준 소모율(CSR)만으로 계획하면 교전 강도가 낮은 날은 과잉 재고가 쌓이고 높은 날은 긴급 재보급이 반복되어 저장 공간과 작업 동선이 낭비된다. 본 프로젝트는 교육용 가상 부대·가상 품목·가상 저장고 자료를 코드로 만들어, 탄약·유류 소모율 데이터 정의와 전처리, CSR 기반 Baseline과 탐색적 분석, 적 저항·화력 계획을 반영한 소모량 예측, 보급 수준(DOS) 연계 재보급 산출, 작전 지도 대시보드와 설명가능성, 저장 배치 최적화와 군수참모 결재선(HITL), 7일 고강도 스트레스 시험, AAR 기반 소모 계수 재보정을 단계별로 구현한다. 최종 목표는 예측 소모량을 근거로 재보급 소요와 저장 위치를 함께 제안해 저장 효율과 작업 동선을 개선하고, 불출·대체 탄종·재배치는 군수참모가 근거를 확인해 결정하도록 지원하는 통합 시뮬레이터를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(군수참모처, 탄약창)에 즉시 도입 가능한 "완성형 국방 AI 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 행정 및 기술 명세서** | 도메인 데이터 명세서(`docs/amm02_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/amm02_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **군수 데이터 정제 모듈** | 10종 정제 룰 및 긴급 불출 보존 ETL 파이프라인(`src/etl/amm02_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **분석 및 통계 검증** | 90일 소모량 Baseline 비교, 가설 검정 및 ABC 파레토 분석(`notebooks/amm02_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 시계열 파생 피처 엔지니어링 모듈(`src/features/amm02_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 재보급 엔진** | GBR+RF 앙상블 및 DOS 재보급 산출 엔진(`src/models/amm02_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | XAI 기여도 분석 및 자연어 보고서 생성기(`src/explainability/amm02_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **저장 최적화 & HITL** | 정수계획 저장 최적화 및 참모 HITL 결재선 엔진(`src/optimization/amm02_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **작전 지도 대시보드** | 전술 격자 지도 연동 인터랙티브 대시보드 웹앱(`src/dashboard/amm02_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 편차 분석, 축소 추정 CSR 재보정 및 3대 정량 효과 리포트(`reports/amm02_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 훈련 데이터 기반 챔피언/챌린저 모델 자동 성능 비교 및 롤오버 파이프라인(`pipeline/amm02_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 적 저항 수준 및 화력 운용 계획 연계 일자별 소모량 시계열 예측 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

작전 계획 연계 일자별 소모량 시계열 예측 모델을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **적 저항 수준(전일 실적)과 화력 운용 계획을 연계한 파생 피처로 일자별 소모량 시계열 예측 모델을 고도화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **전일·최근 평균 소모 비율, CSR 계수, 교전 중 화력 계획, 적 저항 추세 등 사전에 알 수 있는 정보만으로 파생 피처 10개 이상 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/features/amm02_feature_engineering.py 및 기준 모델` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 실무에서 납득 가능한 피처 설계의 논리성

#### 미션 PASS 조건

- 정량 기준: 파생 피처 10개 이상 생성, 기준 모델 MAPE 25% 이하 달성


- 진행 기준: 피처 파이프라인 및 베이스라인 모델 성능 지표 도출


- 검증 방법: 교차 검증(Time-series Cross Validation)을 통한 과적합 방지


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""적 저항 수준(전일 실적)과 화력 운용 계획(당일 계획 화력 임무 수)을 연계한 파생 피처를 만들고, 일자별 탄약·유류 소모량 시계열 예측 모델을 시계열 교차검증으로 평가해 Baseline(CSR 고정 계수)과 비교한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 부대·가상 품목 일일 소모 기록 생성기 (실제 탄약 제원·소모율과 무관)
# 가상 부대 8개 x 90일. 소모량 = 가상 CSR 계수 x 병력 규모 x (교전 강도·적 저항·화력 계획 효과) x 잡음
# ---------------------------------------------------------------------------
UNITS = [f"U{i:02d}" for i in range(1, 9)]
ITEMS = ["ammo_A", "ammo_B", "ammo_C", "ammo_D", "fuel_jet", "fuel_diesel"]
AMMO, FUEL = ITEMS[:4], ITEMS[4:]
PHASES = ["prep", "attack", "defense", "stability"]
CSR = {  # 가상 계수: 병력 100명·1일 기준 소모 단위(탄약=상자, 유류=드럼)
    "prep":      [1.0, 0.6, 0.3, 0.2, 1.5, 6.0],
    "attack":    [6.0, 3.5, 2.0, 1.2, 5.0, 12.0],
    "defense":   [4.5, 3.0, 1.6, 0.8, 3.0, 7.0],
    "stability": [0.8, 0.5, 0.3, 0.2, 1.0, 5.0],
}
PHASE_RESIST = {"prep": 1.0, "attack": 3.2, "defense": 3.6, "stability": 0.8}


def make_ops(n_days=90, seed=7, surge=0):
    """가상 부대별 일일 작전 기록과 품목별 실제 소모량(정답)을 만든다. surge>0이면 마지막 surge일을 고강도 공격 단계로 바꾼다(스트레스 시험용)."""
    rng = np.random.default_rng(seed)
    rows = []
    for uid in UNITS:
        size = int(rng.integers(3, 9)) * 100
        lens = rng.integers(8, 20, 4)
        sched = ["prep"] * lens[0] + ["attack"] * lens[1] + ["defense"] * lens[2] + ["attack"] * lens[3]
        sched = (sched + ["stability"] * n_days)[:n_days]
        if surge:
            sched[-surge:] = ["attack"] * surge
        r = 1.0
        for d, ph in enumerate(sched):
            r = float(np.clip(0.7 * r + 0.3 * (4.8 if surge and d >= n_days - surge else PHASE_RESIST[ph]) + rng.normal(0, 0.35), 0, 5))
            inten = float(np.clip(r / 5 * 0.9 + rng.normal(0, 0.06), 0, 1))
            fire = int(rng.poisson(1 + 8 * inten))
            move = float(rng.gamma(2, {"prep": 4, "attack": 12, "defense": 2, "stability": 5}[ph]))
            weather = str(rng.choice(["clear", "rain", "snow"], p=[0.7, 0.2, 0.1]))
            emerg = bool(rng.random() < 0.02)
            rec = {"unit": uid, "day": d, "phase": ph, "unit_size": size, "intensity": round(inten, 3),
                   "resistance": round(r, 2), "fire_missions": fire, "move_km": round(move, 1),
                   "weather": weather, "emergency": emerg}
            base = size / 100
            for i, it in enumerate(ITEMS):
                c = CSR[ph][i] * base
                if it in AMMO:
                    c *= (0.5 + 0.6 * inten) * (1 + 0.04 * fire * (1.0 if it in ("ammo_C", "ammo_D") else 0.4))
                elif it == "fuel_jet":
                    c *= (0.6 + 0.5 * inten) + 0.03 * fire
                else:
                    c = c * (0.8 + 0.2 * (weather != "clear")) + 0.05 * move * base
                c *= rng.lognormal(0, 0.12) * (2.5 if emerg else 1.0)
                rec[it] = round(float(c), 1)
            rows.append(rec)
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------

# ---- 소모량 예측용 파생 피처: 전일까지의 실적 + 당일 작전·화력 계획(사전에 알 수 있는 값만) ----
FEATS = ["lag1", "lag2", "roll3", "roll7", "csr_base", "size100", "fire_plan", "fire_x_engaged", "resist_lag1",
         "resist_trend", "inten_lag1", "move_plan", "days_in_phase", "engaged", "ph_attack", "ph_defense", "ph_prep",
         "item_idx", "is_fuel"]
DERIVED = ["lag1", "lag2", "roll3", "roll7", "csr_base", "fire_x_engaged", "resist_lag1", "resist_trend",
           "inten_lag1", "days_in_phase", "engaged", "ph_attack", "ph_defense", "ph_prep"]


def build_long(ops):
    """부대x일자 기록을 (부대, 일자, 품목) 행으로 펼치고 파생 피처를 만든다."""
    # [TODO : 학생 실습 4] 적 저항 수준 및 화력 운용 계획 연계 일자별 소모량 시계열 예측 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def to_qty(d, y_pred):
    """예측 보정 계수(log)를 소모량으로 되돌린다: 기준량 x exp(예측)."""
    # [TODO : 학생 실습 4] 적 저항 수준 및 화력 운용 계획 연계 일자별 소모량 시계열 예측 모델 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def mape(y, p):
    return float(np.mean(np.abs(np.asarray(p) - np.asarray(y)) / np.asarray(y)))


data = build_long(make_ops(90, seed=7)).sort_values(["day", "unit", "item"]).reset_index(drop=True)
print(f"학습 행 {len(data)}개(부대 8 x 품목 6 x 일자), 입력 피처 {len(FEATS)}개 중 파생 피처 {len(DERIVED)}개 (목표 10개 이상: {'달성' if len(DERIVED) >= 10 else '미달'})")

# 1) 시계열 교차검증: 과거 일자로 학습하고 이후 일자를 예측(미래 정보 누설 방지)
days = np.sort(data["day"].unique())
rows = []
for k, (tr_d, te_d) in enumerate(TimeSeriesSplit(n_splits=5).split(days), start=1):
    tr, te = data[data["day"].isin(days[tr_d])], data[data["day"].isin(days[te_d])]
    m = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, subsample=0.8, random_state=0)
    m.fit(tr[FEATS], tr["y"])
    pred = to_qty(te, m.predict(te[FEATS]))
    base = te["csr_base"] * te["size100"]
    rows.append({"폴드": k, "검증 일자": f"{days[te_d][0]}~{days[te_d][-1]}", "Baseline MAPE": mape(te["qty"], base),
                 "모델 MAPE": mape(te["qty"], pred), "전일 비율 유지 MAPE": mape(te["qty"], te["lag1"] * te["csr_base"] * te["size100"])})
cv = pd.DataFrame(rows)
print("\n[시계열 교차검증 결과]")
print(cv.to_string(index=False, float_format=lambda x: f"{x:.1%}"))
avg = cv["모델 MAPE"].mean()
print(f"평균 MAPE: Baseline {cv['Baseline MAPE'].mean():.1%} -> 모델 {avg:.1%} (목표 25% 이하: {'달성' if avg <= 0.25 else '미달'})")

# 2) 최종 모델: 앞 70일 학습, 마지막 18일 검증
tr, te = data[data["day"] < 72], data[data["day"] >= 72]
model = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, subsample=0.8, random_state=0).fit(tr[FEATS], tr["y"])
te = te.assign(pred=to_qty(te, model.predict(te[FEATS])), base=te["csr_base"] * te["size100"])
by_item = te.groupby("item")[["qty", "pred", "base"]].apply(lambda g: pd.Series({"Baseline": mape(g["qty"], g["base"]), "모델": mape(g["qty"], g["pred"])}))
print("\n[품목별 검증 MAPE(마지막 18일)]")
print(by_item.T.to_string(float_format=lambda x: f"{x:.1%}"))

# 3) 피처 중요도(차원 축소 후보 확인)
imp = pd.Series(model.feature_importances_, index=FEATS).sort_values(ascending=False)
print("\n피처 중요도 상위 6:", ", ".join(f"{k}({v:.2f})" for k, v in imp.head(6).items()))
print("중요도 0.01 미만(제거 후보):", ", ".join(imp[imp < 0.01].index) or "없음")

# 4) 시각화
full = data.assign(pred=to_qty(data, model.predict(data[FEATS])))
u = full[(full["unit"] == "U01") & (full["item"] == "ammo_A")]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(u["day"], u["qty"], label="actual"); ax[0].plot(u["day"], u["pred"], label="model")
ax[0].plot(u["day"], u["csr_base"] * u["size100"], ls="--", label="baseline (CSR)"); ax[0].axvline(72, c="gray", ls=":")
ax[0].set_title("U01 ammo_A daily forecast (right of dotted line = hold-out)"); ax[0].set_xlabel("day"); ax[0].legend()
imp.head(10)[::-1].plot.barh(ax=ax[1], title="Feature importance (top 10)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
학습 행 4176개(부대 8 x 품목 6 x 일자), 입력 피처 19개 중 파생 피처 14개 (목표 10개 이상: 달성)

[시계열 교차검증 결과]
 폴드 검증 일자  Baseline MAPE  모델 MAPE  전일 비율 유지 MAPE
  1 20~33          15.4%    15.4%          23.0%
  2 34~47          17.7%    19.0%          20.4%
  3 48~61          33.4%    13.0%          17.3%
  4 62~75          51.8%    13.6%          21.7%
  5 76~89          48.7%    13.3%          21.8%
평균 MAPE: Baseline 33.4% -> 모델 14.9% (목표 25% 이하: 달성)

[품목별 검증 MAPE(마지막 18일)]
item      ammo_A  ammo_B  ammo_C  ammo_D  fuel_diesel  fuel_jet
Baseline   66.2%   64.7%   60.0%   55.2%        15.4%     36.6%
모델         12.7%   12.3%   13.0%   13.6%        13.5%     12.6%

피처 중요도 상위 6: roll3(0.25), resist_lag1(0.25), csr_base(0.09), fire_x_engaged(0.09), fire_plan(0.07), roll7(0.04)
중요도 0.01 미만(제거 후보): lag2, ph_defense, ph_attack, engaged, ph_prep
```
</details>
